# Customer Churn - Feature Engineering

**Goal**: Create new features that better understand patterns in the data and improve the performance of the ML models.

As identified during the EDA, customers more likely to churn tend to present the following characteristics:
- short tenure
- high monthly charges
- month-to-month contracts

Creating features that help capture these patterns may improve the model’s ability to identify customers at higher risk of churn.

## Load Dataset

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_data, plot_feature, plot_vs
from src.data.preprocess import encode_categorical

df = load_data('../data/processed/telco_clean.csv')

## Tenure Groups

Based on the previous analysis, tenure ranges from 0 to 72 months (6 years) and presents a bimodal distribution, with a high concentration of customers at very low tenure and another cluster at longer tenure. To better capture these patterns, tenure is divided into four groups:
- 0 - 6 months
- 6 - 24 months
- 24 - 60 months
- 60 + months

These groups are defined to reflect the observed concentration of customers in early and long tenure periods.

In [2]:
df['TenureGroups'] = pd.cut(
    df['tenure'], 
    bins=[0, 6, 24, 60, np.inf],
    labels=['0-6', '6-24', '24-60', '60+'],
    include_lowest=True
    )

# Drop the original 'tenure' column as it's now represented in 'TenureGroups'
df.drop(columns=['tenure'], inplace=True)

df['TenureGroups'].value_counts()

24-60    2426
6-24     1729
0-6      1481
60+      1407
Name: TenureGroups, dtype: int64

Since this new feature created is a categorical variable, it's necessary to use one-hot encoding to make it easir to apply the ML models.

In [3]:
df, categorical_columns = encode_categorical(df)

df.head()

,gender,SeniorCitizen,Partner,Dependents,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,MultipleLines_No phone service,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroups_6-24,TenureGroups_24-60,TenureGroups_60+
0,0,0,1,0,0,1,29,29,0,1,...,0,0,0,0,0,1,0,0,0,0
1,1,0,0,0,1,0,56,1889,0,0,...,0,0,1,0,0,0,1,0,1,0
2,1,0,0,0,1,1,53,108,1,0,...,0,0,0,0,0,0,1,0,0,0
3,1,0,0,0,0,0,42,1840,0,1,...,0,0,1,0,0,0,0,0,1,0
4,0,0,0,0,1,1,70,151,1,0,...,0,0,0,0,0,1,0,0,0,0


## Service Count

This dataset includes information on the different services each customer has subscribed to, such as phone service, multiple lines, internet service, online security, online backup, device protection, tech support, and streaming services (TV and movies).

It may be useful to quantify the total number of services per customer, as this can reflect the level of engagement with the company. Customers subscribed to more services may be less likely to churn, while those with fewer services may be more likely.

To capture this behaviour, a new feature will be created representing the total number of services each customer has subscribed to.

In [4]:
# Creating a new column representing costumers with either fiber optic or DSL internet service
df['InternetService_Yes'] = 1 - df['InternetService_No']

service_columns = [col for col in df.columns if col.endswith('_Yes')]

df['ServiceCount'] = 0

for service in service_columns:
    df['ServiceCount'] += df[service]

df['ServiceCount'].value_counts()

0    1184
4     978
3     957
5     933
1     825
2     816
6     722
7     420
8     208
Name: ServiceCount, dtype: int64

## Save Clean Dataset

In [5]:
df.to_csv("../data/processed/telco_feature_engineering.csv", index=False)